# ISIC 2024 `strict_input + iddx_full` Dataset Contract

Updated: 2026-05-14

이 노트북은 ISIC 2024 `train-metadata.csv` 전체를 `official_train_pool`로 두고, patient-level Triple Stratified Nested CV strict input 계약을 정리한다. 기존 `20260427` 노트북의 실행 가능한 점검 흐름은 유지하되, 별도 고정 test split을 먼저 떼어낸 뒤 남은 데이터로 CV를 수행하는 설명은 더 이상 기본 protocol로 쓰지 않는다.


## 0. 개요

### 0.1 이 노트북의 범위

1. 데이터와 누수 위험 요약
2. 컬럼 role registry
3. `strict_input` 계약
4. `iddx_full` train-only sidecar 계약
5. Label Embedding / Graph Metric Learning 후보 계약
6. `official_train_pool` 기반 Nested CV split terminology and audit
7. Dataset / training pipeline handoff checklist

### 0.2 핵심 결론

메인 논문 실험 입력은 다음으로 고정한다.

```text
image + ordinary inference-time tabular metadata -> malignant probability
```

`iddx_full`은 ordinary inference-time input이 아니다. `iddx_full` 또는 이를 가공한 label/text signal은 candidate-only privileged supervision으로만 관리한다.

### 0.3 이 노트북에서 하지 않는 것

- 산출물 CSV 저장
- reusable training code 작성
- hyperparameter search
- threshold 선택
- public test metadata를 paper-facing 평가로 사용

이 노트북은 계약과 audit용 evidence를 만드는 문서형 notebook이다. 원자료는 읽기 전용으로만 다룬다.

### 0.4 용어 통일

| 용어 | 의미 |
|---|---|
| `official_train_pool` | `data/raw/isic-2024-challenge/train-metadata.csv` 전체 |
| `cv_test_fold` | outer fold id이며 `outer_test`와 같은 의미 |
| `outer_test` | 현재 outer fold의 최종 평가 전용 patient-disjoint partition |
| `cv_train` | 현재 `outer_test`를 제외한 outer train pool |
| `inner_validation` | `cv_train` 내부에서 model choice, hyperparameter, early stopping, threshold, calibration만을 위해 분리하는 validation subset |
| `inner_train` | `inner_validation`까지 제외한 model/preprocessing/class-weight fit subset |
| `strict_input` | inference 시점에 사용 가능한 ordinary tabular metadata |
| `iddx_full_train_only` | candidate-only privileged sidecar |

### 0.5 이 노트북에서 `계약`의 의미

여기서 `계약`은 법적 의미가 아니라, 후속 Dataset, preprocessing, training, inference 코드가 반드시 지켜야 하는 **데이터 사용 규칙과 입출력 약속**을 뜻한다. 예를 들어 어떤 컬럼을 model input으로 쓸 수 있는지, 어떤 컬럼은 train-only sidecar인지, 어떤 preprocessing은 fold-local train에서만 fit해야 하는지, 어떤 결과가 paper-facing evidence로 인정되는지를 명확히 고정하는 것이다.

따라서 `strict_input 계약`은 "이 컬럼들은 inference 시점에도 사용할 수 있는 ordinary tabular input"이라는 뜻이고, `iddx_full train-only sidecar 계약`은 "이 값은 후보 실험의 학습 보조 신호로만 쓰며 validation/test/inference 입력으로 요구하면 안 된다"는 뜻이다.


## 1. 데이터와 위험 요약

이 장은 계약 문서에 필요한 최소 데이터 사실만 확인한다. `train-metadata.csv` 전체를 `official_train_pool`로 사용하고, 별도 고정 test split을 먼저 만들지 않고 nested CV의 outer_test를 최종 평가 단위로 둔다.


In [1]:
# 노트북 실행 위치가 달라도 프로젝트 패키지와 원본 metadata를 안정적으로 찾기 위한 초기 설정이다.
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# IPython이 없는 배치 실행 환경에서도 display 호출이 실패하지 않도록 fallback을 둔다.
try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def Markdown(value):
        return value

    def display(value):
        print(value)


# cwd와 상위 경로를 훑어 repo root를 찾고 src 패키지를 import path에 추가한다.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'notebooks/isic_2024').exists()
        and (candidate / 'src/isic2024_multimodal').exists()
    ),
    Path.cwd(),
)
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Jupyter 커널이 이전 버전의 로컬 모듈을 캐시할 수 있으므로 명시적으로 reload한다.
# 특히 split helper를 수정한 직후 커널 재시작 없이 이 노트북을 다시 실행할 때 필요하다.
import importlib  # noqa: E402
import isic2024_multimodal.data.triple_stratified_split as triple_split  # noqa: E402

triple_split = importlib.reload(triple_split)
assign_triple_stratified_groups = triple_split.assign_triple_stratified_groups
build_nested_cv_assignments = triple_split.build_nested_cv_assignments
make_patient_split_profile = triple_split.make_patient_split_profile


# raw metadata는 읽기 전용으로만 사용하며, 이후 분석은 official_train_pool 복사본에서 수행한다.
METADATA_DIR_CANDIDATES = [
    PROJECT_ROOT / 'data/raw/isic-2024-challenge',
    PROJECT_ROOT / 'data/raw/isic_2024_challenge',
    PROJECT_ROOT / 'data/raw',
]
METADATA_DIR = next(
    (candidate for candidate in METADATA_DIR_CANDIDATES if (candidate / 'train-metadata.csv').exists()),
    None,
)
if METADATA_DIR is None:
    checked_paths = '\n'.join(str(path) for path in METADATA_DIR_CANDIDATES)
    raise FileNotFoundError('Could not find ISIC 2024 train-metadata.csv under one of:\n' + checked_paths)

TRAIN_PATH = METADATA_DIR / 'train-metadata.csv'
PUBLIC_TEST_PATH = METADATA_DIR / 'test-metadata.csv'
official_train_pool = pd.read_csv(TRAIN_PATH, low_memory=False)
public_test_df = pd.read_csv(PUBLIC_TEST_PATH, low_memory=False) if PUBLIC_TEST_PATH.exists() else None
df = official_train_pool.copy()
target_name_map = {0: 'benign', 1: 'malignant'}


def display_table_with_note(title, note, table):
    # 각 표가 단순 출력이 아니라 어떤 protocol 판단을 지원하는지 짧은 문단으로 함께 보여준다.
    display(Markdown(title))
    display(Markdown(note))
    display(table)


# 표 1은 데이터 규모, target 희귀성, patient-level grouping 위험을 빠르게 확인하는 요약이다.
summary_df = pd.DataFrame(
    [
        {'item': 'source_path', 'value': TRAIN_PATH.as_posix()},
        {'item': 'official_train_pool_rows', 'value': len(official_train_pool)},
        {'item': 'official_train_pool_patients', 'value': official_train_pool['patient_id'].nunique()},
        {'item': 'lesion_id_non_null', 'value': int(official_train_pool['lesion_id'].notna().sum())},
        {'item': 'positive_rows', 'value': int(official_train_pool['target'].sum())},
        {'item': 'positive_rate_pct', 'value': round(float(official_train_pool['target'].mean() * 100), 5)},
        {'item': 'public_test_rows_schema_only', 'value': len(public_test_df) if public_test_df is not None else pd.NA},
    ]
)

summary_item_description_map = {
    'source_path': '분석에 사용한 원본 train metadata 경로',
    'official_train_pool_rows': 'official_train_pool에 포함된 전체 sample(row) 수',
    'official_train_pool_patients': 'official_train_pool에 포함된 고유 patient_id 수',
    'lesion_id_non_null': 'lesion_id가 비어 있지 않은 row 수; lesion alignment 가능 범위 확인',
    'positive_rows': 'target=1 malignant row 수',
    'positive_rate_pct': '전체 row 중 malignant 비율; ultra-rare target 정도 확인',
    'public_test_rows_schema_only': 'public test metadata row 수; label 없는 schema 확인용으로만 취급',
}
summary_df['item_description'] = summary_df['item'].map(summary_item_description_map)
summary_df = summary_df[['item', 'item_description', 'value']]

patient_target_summary_df = official_train_pool.groupby('patient_id').agg(
    n_rows=('isic_id', 'size'),
    positive_rows=('target', 'sum'),
    lesion_id_non_null=('lesion_id', lambda s: int(s.notna().sum())),
).reset_index()
patient_target_summary_df['has_malignant'] = patient_target_summary_df['positive_rows'] > 0
patient_risk_summary_df = pd.DataFrame(
    [
        {'item': 'patients_with_malignant', 'value': int(patient_target_summary_df['has_malignant'].sum())},
        {'item': 'max_rows_per_patient', 'value': int(patient_target_summary_df['n_rows'].max())},
        {'item': 'median_rows_per_patient', 'value': round(float(patient_target_summary_df['n_rows'].median()), 3)},
    ]
)

display_table_with_note(
    '**표 1-a. 데이터 규모와 target 희귀성**',
    '이 표는 원본 train metadata의 규모와 malignant target 희귀성을 확인한다. positive_rows와 positive_rate_pct가 낮기 때문에 accuracy만으로 모델 품질을 판단하면 안 된다.',
    summary_df,
)
display_table_with_note(
    '**표 1-b. patient-level grouping 위험 요약**',
    '이 표는 한 환자가 여러 sample을 가질 수 있는 구조와 malignant 환자 수를 보여준다. row-level random split을 피하고 patient_id 기준 split을 사용해야 하는 이유를 확인한다.',
    patient_risk_summary_df,
)


**표 1-a. 데이터 규모와 target 희귀성**

이 표는 원본 train metadata의 규모와 malignant target 희귀성을 확인한다. positive_rows와 positive_rate_pct가 낮기 때문에 accuracy만으로 모델 품질을 판단하면 안 된다.

,item,item_description,value
0,source_path,분석에 사용한 원본 train metadata 경로,/home/ubuntu/wksp/paper_ajou_dev/data/raw/isic...
1,official_train_pool_rows,official_train_pool에 포함된 전체 sample(row) 수,401059
2,official_train_pool_patients,official_train_pool에 포함된 고유 patient_id 수,1042
3,lesion_id_non_null,lesion_id가 비어 있지 않은 row 수; lesion alignment 가능...,22058
4,positive_rows,target=1 malignant row 수,393
5,positive_rate_pct,전체 row 중 malignant 비율; ultra-rare target 정도 확인,0.09799
6,public_test_rows_schema_only,public test metadata row 수; label 없는 schema 확인...,3


**표 1-b. patient-level grouping 위험 요약**

이 표는 한 환자가 여러 sample을 가질 수 있는 구조와 malignant 환자 수를 보여준다. row-level random split을 피하고 patient_id 기준 split을 사용해야 하는 이유를 확인한다.

,item,value
0,patients_with_malignant,259.0
1,max_rows_per_patient,9184.0
2,median_rows_per_patient,241.5


### 1.1 해석

ISIC2024 malignant target은 row 기준으로 매우 희귀하다. 따라서 row-level random split은 paper-facing 실험으로 부적절하고, 모든 비교는 `patient_id -> lesion_id -> isic_id` grouping을 보존해야 한다. `public test metadata`는 label이 없거나 schema 확인에 가깝기 때문에 내부 검증 evidence로 쓰지 않는다.


## 2. 컬럼 role registry

이 장은 각 컬럼이 어떤 regime에 속하는지 고정한다. 목적은 입력 후보를 늘리는 것이 아니라, inference-time input과 train-only/reference-only 신호를 분리하는 것이다.


In [2]:
# 각 컬럼을 strict inference input, train-only privileged signal, reference, identifier로 분리한다.
REGIME_DESCRIPTION_MAP = {
    'strict': 'inference 시점에도 사용할 수 있는 ordinary tabular metadata',
    'schema_constant': 'schema 확인용 constant metadata이며 strict_input에서는 제외',
    'relaxed': '민감도 분석에서만 선택적으로 볼 수 있는 context field',
    'oracle': 'iddx_full처럼 후보 실험의 train-only privileged signal',
    'reference': '진단/병리 관련 reference field이며 ordinary input 금지',
    'not_feature': 'sample 정렬과 patient-level grouping을 위한 identifier',
    'label': 'supervised target label',
    'unassigned': '현재 계약에 명시되지 않은 컬럼; 사용 전 검토 필요',
}

REGIME_DISPLAY_MAP = {
    'strict': 'strict_input',
    'schema_constant': 'Schema-only constant metadata / excluded from model input',
    'relaxed': 'Relaxed context optional sensitivity check',
    'oracle': 'Train-only iddx_full text',
    'reference': 'Reference only / excluded reference fields',
    'not_feature': 'Identifier / grouping key',
    'label': 'Label',
}

RELAXED_EXTRA_COLS = ['attribution', 'copyright_license']

# iddx_full은 ordinary input이 아니라 후보 실험에서만 쓰는 train-only sidecar로 고정한다.
TRAIN_ONLY_IDDX_FULL_TEXT_COLS = ['iddx_full']
REFERENCE_ONLY_COLS = [
    'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5',
    'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence',
]
SCHEMA_ONLY_CONSTANT_COLS = ['image_type']
IDENTIFIER_GROUPING_COLS = ['isic_id', 'patient_id', 'lesion_id']
ORDINARY_METADATA_COLS = [
    'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm',
    'image_type', 'tbp_tile_type', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B',
    'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext',
    'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio',
    'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL',
    'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location',
    'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence',
    'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM',
    'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt',
    'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z',
]

# strict_input은 inference-time ordinary metadata에서 schema-only constant를 제외한 컬럼 집합이다.
STRICT_COLS = [column for column in ORDINARY_METADATA_COLS if column not in SCHEMA_ONLY_CONSTANT_COLS]

def assign_regime(column):
    if column == 'target':
        return 'label'
    if column in IDENTIFIER_GROUPING_COLS:
        return 'not_feature'
    if column in TRAIN_ONLY_IDDX_FULL_TEXT_COLS:
        return 'oracle'
    if column in REFERENCE_ONLY_COLS:
        return 'reference'
    if column in RELAXED_EXTRA_COLS:
        return 'relaxed'
    if column in SCHEMA_ONLY_CONSTANT_COLS:
        return 'schema_constant'
    if column in STRICT_COLS:
        return 'strict'
    return 'unassigned'


# 실제 train metadata의 모든 컬럼을 role registry로 변환해 누락/오분류를 눈으로 검토한다.
registry_rows = []
for column in df.columns:
    regime = assign_regime(column)
    registry_rows.append(
        {
            'column': column,
            'regime': regime,
            'regime_display': REGIME_DISPLAY_MAP.get(regime, 'Unassigned'),
            'dtype': str(df[column].dtype),
            'null_ratio_pct': round(float(df[column].isna().mean() * 100), 4),
            'n_unique': int(df[column].nunique(dropna=True)),
        }
    )
column_registry_df = pd.DataFrame(registry_rows)
regime_summary_df = (
    column_registry_df.groupby(['regime', 'regime_display'], dropna=False)
    .size()
    .reset_index(name='column_count')
    .sort_values(['regime', 'regime_display'])
    .reset_index(drop=True)
)
regime_summary_df['regime_description'] = regime_summary_df['regime'].map(REGIME_DESCRIPTION_MAP)
regime_summary_df = regime_summary_df[['regime', 'regime_display', 'regime_description', 'column_count']]

# 계약에 필요한 컬럼이 없으면 downstream dataset 정의 전에 즉시 중단한다.
missing_registry_columns = sorted(
    set(ORDINARY_METADATA_COLS + RELAXED_EXTRA_COLS + TRAIN_ONLY_IDDX_FULL_TEXT_COLS + REFERENCE_ONLY_COLS + IDENTIFIER_GROUPING_COLS + ['target'])
    - set(df.columns)
)
if missing_registry_columns:
    raise KeyError(f'계약 컬럼이 train metadata에 없습니다: {missing_registry_columns}')

display_table_with_note(
    '**표 2-a. regime별 컬럼 수**',
    '이 표는 metadata 컬럼이 strict input, train-only signal, reference, identifier 등으로 어떻게 나뉘는지 요약한다. oracle/reference/not_feature 컬럼이 ordinary baseline input에 섞이지 않았는지 확인하는 첫 단계다.',
    regime_summary_df,
)
display_table_with_note(
    '**표 2-b. role registry**',
    '이 표는 원본 metadata의 각 컬럼을 하나씩 role에 매핑한 registry다. dtype, null ratio, unique count를 함께 보면서 입력 가능 여부와 후속 preprocessing 대상을 확인한다.',
    column_registry_df.sort_values(['regime', 'column']).reset_index(drop=True),
)


**표 2-a. regime별 컬럼 수**

이 표는 metadata 컬럼이 strict input, train-only signal, reference, identifier 등으로 어떻게 나뉘는지 요약한다. oracle/reference/not_feature 컬럼이 ordinary baseline input에 섞이지 않았는지 확인하는 첫 단계다.

,regime,regime_display,regime_description,column_count
0,label,Label,supervised target label,1
1,not_feature,Identifier / grouping key,sample 정렬과 patient-level grouping을 위한 identifier,3
2,oracle,Train-only iddx_full text,iddx_full처럼 후보 실험의 train-only privileged signal,1
3,reference,Reference only / excluded reference fields,진단/병리 관련 reference field이며 ordinary input 금지,8
4,relaxed,Relaxed context optional sensitivity check,민감도 분석에서만 선택적으로 볼 수 있는 context field,2
5,schema_constant,Schema-only constant metadata / excluded from ...,schema 확인용 constant metadata이며 strict_input에서는 제외,1
6,strict,strict_input,inference 시점에도 사용할 수 있는 ordinary tabular metadata,39


**표 2-b. role registry**

이 표는 원본 metadata의 각 컬럼을 하나씩 role에 매핑한 registry다. dtype, null ratio, unique count를 함께 보면서 입력 가능 여부와 후속 preprocessing 대상을 확인한다.

,column,regime,regime_display,dtype,null_ratio_pct,n_unique
0,target,label,Label,int64,0.0000,2
1,isic_id,not_feature,Identifier / grouping key,object,0.0000,401059
2,lesion_id,not_feature,Identifier / grouping key,object,94.5001,22058
3,patient_id,not_feature,Identifier / grouping key,object,0.0000,1042
4,iddx_full,oracle,Train-only iddx_full text,object,0.0000,52
5,iddx_1,reference,Reference only / excluded reference fields,object,0.0000,3
6,iddx_2,reference,Reference only / excluded reference fields,object,99.7337,14
7,iddx_3,reference,Reference only / excluded reference fields,object,99.7345,25
8,iddx_4,reference,Reference only / excluded reference fields,object,99.8626,26
9,iddx_5,reference,Reference only / excluded reference fields,object,99.9998,1


### 2.1 해석

`strict_input`은 ordinary inference-time metadata이고, `iddx_full`과 diagnosis/reference column은 기본 입력에서 제외한다. `lesion_id`는 feature가 아니라 grouping/alignment identifier로만 보존한다.


## 3. `strict_input` 계약

`strict_input`은 `inner_train`, `inner_validation`, `outer_test`, inference 시점에 요구할 수 있는 ordinary tabular metadata model input이다. 이 장은 컬럼 목록과 결측 profile만 고정하며, imputation/scaling/encoding은 fold-local training code에서만 fit한다.


In [3]:
# strict_input profile은 downstream dataset이 inference-time tabular input으로 받아야 할 계약이다.
strict_input_profile_df = column_registry_df.loc[column_registry_df['column'].isin(STRICT_COLS)].copy()
strict_input_profile_df['input_contract'] = 'ordinary inference-time tabular input'
# image_type 같은 schema-only constant metadata는 기록만 남기고 strict_input에서는 제외한다.
schema_constant_profile_df = column_registry_df.loc[column_registry_df['column'].isin(SCHEMA_ONLY_CONSTANT_COLS)].copy()
schema_constant_profile_df['input_contract'] = 'schema-only constant metadata; excluded from strict_input'
# 여기서는 dtype별 컬럼 수만 요약한다. 실제 imputer/encoder fit은 fold-local train subset에서만 수행해야 한다.
strict_input_numeric_cols = [column for column in STRICT_COLS if pd.api.types.is_numeric_dtype(df[column])]
strict_input_categorical_cols = [column for column in STRICT_COLS if column not in strict_input_numeric_cols]
strict_input_summary_df = pd.DataFrame(
    [
        {'item': 'strict_input_columns', 'value': len(STRICT_COLS)},
        {'item': 'schema_only_constant_columns_excluded', 'value': len(SCHEMA_ONLY_CONSTANT_COLS)},
        {'item': 'numeric_columns', 'value': len(strict_input_numeric_cols)},
        {'item': 'categorical_columns', 'value': len(strict_input_categorical_cols)},
        {'item': 'max_null_ratio_pct', 'value': round(float(strict_input_profile_df['null_ratio_pct'].max()), 4)},
    ]
)

display_table_with_note(
    '**표 3-a. strict_input 요약**',
    '이 표는 inference 시점에도 사용할 수 있는 strict_input의 규모와 타입 구성을 요약한다. 실제 imputation, scaling, encoding은 여기서 fit하지 않고 fold-local train에서만 fit한다.',
    strict_input_summary_df,
)
display_table_with_note(
    '**표 3-b. strict_input 컬럼 계약**',
    '이 표는 strict tabular baseline과 multimodal baseline이 사용할 수 있는 ordinary metadata 컬럼 목록이다. 각 컬럼의 dtype과 결측률을 보고 fold-local preprocessing 대상을 정한다.',
    strict_input_profile_df[['column', 'dtype', 'null_ratio_pct', 'n_unique', 'input_contract']].reset_index(drop=True),
)
display_table_with_note(
    '**표 3-c. schema-only constant metadata 제외 계약**',
    '이 표는 schema 확인에는 필요하지만 strict_input feature에서는 제외할 컬럼을 분리한다. constant 성격의 metadata가 baseline feature 수에 섞이지 않도록 명시한다.',
    schema_constant_profile_df[['column', 'dtype', 'null_ratio_pct', 'n_unique', 'input_contract']].reset_index(drop=True),
)


**표 3-a. strict_input 요약**

이 표는 inference 시점에도 사용할 수 있는 strict_input의 규모와 타입 구성을 요약한다. 실제 imputation, scaling, encoding은 여기서 fit하지 않고 fold-local train에서만 fit한다.

,item,value
0,strict_input_columns,39.0000
1,schema_only_constant_columns_excluded,1.0000
2,numeric_columns,34.0000
3,categorical_columns,5.0000
4,max_null_ratio_pct,2.8716


**표 3-b. strict_input 컬럼 계약**

이 표는 strict tabular baseline과 multimodal baseline이 사용할 수 있는 ordinary metadata 컬럼 목록이다. 각 컬럼의 dtype과 결측률을 보고 fold-local preprocessing 대상을 정한다.

,column,dtype,null_ratio_pct,n_unique,input_contract
0,age_approx,float64,0.6977,16,ordinary inference-time tabular input
1,sex,object,2.8716,2,ordinary inference-time tabular input
2,anatom_site_general,object,1.4352,5,ordinary inference-time tabular input
3,clin_size_long_diam_mm,float64,0.0000,1758,ordinary inference-time tabular input
4,tbp_tile_type,object,0.0000,2,ordinary inference-time tabular input
5,tbp_lv_A,float64,0.0000,386052,ordinary inference-time tabular input
6,tbp_lv_Aext,float64,0.0000,385304,ordinary inference-time tabular input
7,tbp_lv_B,float64,0.0000,389890,ordinary inference-time tabular input
8,tbp_lv_Bext,float64,0.0000,387763,ordinary inference-time tabular input
9,tbp_lv_C,float64,0.0000,390703,ordinary inference-time tabular input


**표 3-c. schema-only constant metadata 제외 계약**

이 표는 schema 확인에는 필요하지만 strict_input feature에서는 제외할 컬럼을 분리한다. constant 성격의 metadata가 baseline feature 수에 섞이지 않도록 명시한다.

,column,dtype,null_ratio_pct,n_unique,input_contract
0,image_type,object,0.0,1,schema-only constant metadata; excluded from s...


### 3.1 해석

`strict_input`은 원시 metadata 중심의 일반 model input 계약이다. 결측 처리, scaling, encoding, class weight, sampling plan은 전체 dataframe에서 fit하지 않고, 각 nested split의 `inner_train` 또는 해당 training subset에서만 fit해야 한다.


## 4. `iddx_full` train-only sidecar 계약

`iddx_full`은 ordinary tabular feature가 아니다. 같은 `isic_id` sample에 연결된 train-only text field로 추적하며, tokenizer vocabulary, label grouping, privileged teacher, auxiliary loss 같은 candidate 실험에서만 사용할 수 있다.


In [4]:
# sample-level key, target, strict_input, iddx_full sidecar가 모두 존재하는지 먼저 검사한다.
required_sample_columns = ['isic_id', 'patient_id', 'lesion_id', 'target', *STRICT_COLS, 'iddx_full']
missing_sample_columns = [column for column in required_sample_columns if column not in df.columns]
if missing_sample_columns:
    raise KeyError(f'sample-level contract 컬럼이 없습니다: {missing_sample_columns}')


# iddx_full은 존재 여부만 확인하며, inference/scoring 입력으로 승격하지 않는다.
sample_alignment_check_df = pd.DataFrame(
    [
        {'check': 'isic_id_unique', 'pass': bool(df['isic_id'].is_unique), 'value': int(df['isic_id'].nunique())},
        {'check': 'patient_id_present', 'pass': bool(df['patient_id'].notna().all()), 'value': int(df['patient_id'].isna().sum())},
        {'check': 'target_binary', 'pass': set(df['target'].dropna().unique()).issubset({0, 1}), 'value': sorted(df['target'].dropna().unique().tolist())},
        {'check': 'iddx_full_present', 'pass': bool(df['iddx_full'].notna().all()), 'value': int(df['iddx_full'].isna().sum())},
    ]
)


# phase별 입력 계약은 iddx_full-derived signal이 train-only candidate branch에만 머무르도록 명시한다.
phase_input_contract_df = pd.DataFrame(
    [
        {
            'phase': 'inner_train',
            'strict_input': 'required',
            'iddx_full_or_derived': 'optional candidate auxiliary signal',
            'allowed_use': 'candidate-only training objective inside the fold-local training subset',
        },
        {
            'phase': 'inner_validation',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not used as model input, scoring feature, threshold feature, or calibration feature',
            'allowed_use': 'leakage audit only',
        },
        {
            'phase': 'cv_test_fold',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not allowed for model choice, threshold selection, calibration, or scoring',
            'allowed_use': 'none for paper-facing evaluation',
        },
        {
            'phase': 'inference',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not required',
            'allowed_use': 'none',
        },
    ]
)

display_table_with_note(
    '**표 4-a. sample alignment checks**',
    '이 표는 sample key, patient key, binary target, iddx_full sidecar가 row 단위로 정렬되어 있는지 확인한다. iddx_full 존재 여부는 확인하지만 inference input으로 쓰겠다는 뜻은 아니다.',
    sample_alignment_check_df,
)
display_table_with_note(
    '**표 4-b. phase별 입력 계약**',
    '이 표는 inner_train, inner_validation, outer_test, inference 단계별 허용 입력을 분리한다. iddx_full-derived signal은 candidate training branch에만 허용되고 scoring과 inference에서는 금지된다.',
    phase_input_contract_df,
)


**표 4-a. sample alignment checks**

이 표는 sample key, patient key, binary target, iddx_full sidecar가 row 단위로 정렬되어 있는지 확인한다. iddx_full 존재 여부는 확인하지만 inference input으로 쓰겠다는 뜻은 아니다.

,check,pass,value
0,isic_id_unique,True,401059
1,patient_id_present,True,0
2,target_binary,True,"[0, 1]"
3,iddx_full_present,True,0


**표 4-b. phase별 입력 계약**

이 표는 inner_train, inner_validation, outer_test, inference 단계별 허용 입력을 분리한다. iddx_full-derived signal은 candidate training branch에만 허용되고 scoring과 inference에서는 금지된다.

,phase,strict_input,iddx_full_or_derived,allowed_use
0,inner_train,required,optional candidate auxiliary signal,candidate-only training objective inside the f...
1,inner_validation,required,"not used as model input, scoring feature, thre...",leakage audit only
2,cv_test_fold,required,"not allowed for model choice, threshold select...",none for paper-facing evaluation
3,inference,required,not required,none


### 4.1 해석

`iddx_full`은 row-aligned sidecar로는 보존할 수 있지만, v0 strict baseline이나 multimodal inference input에는 들어가지 않는다. candidate 실험을 하더라도 `inner_validation`, `outer_test`, inference path는 `image + strict_input`만 요구해야 한다.


## 5. Label Embedding / Graph Metric Learning 후보 전처리 계약

이 후보는 `iddx_full`을 직접 inference feature로 쓰지 않고, train-only auxiliary label 또는 prototype/metric signal로만 쓰는 방향이다. 아직 기본 baseline이 아니며, LUPI/privileged supervision candidate로만 표시한다.


In [5]:
# iddx_full 기반 label group은 LUPI/privileged supervision 후보 실험용 계약이며 main input이 아니다.
LABEL_EMBEDDING_GROUPS = ['MEL', 'BCC', 'SCC', 'NV', 'OTHER']
LABEL_METRIC_ACTIVE_GROUPS = ['MEL', 'BCC', 'SCC', 'NV']
OTHER_AUX_WEIGHT_DEFAULT = 0.05


# 문자열 규칙으로 MEL/BCC/SCC/NV/OTHER 후보 보조 라벨을 만든다. 모델 입력 feature가 아니다.
def map_iddx_full_to_label_group(value):
    text = str(value)
    if 'Basal cell carcinoma' in text:
        return 'BCC'
    if 'Squamous cell carcinoma' in text:
        return 'SCC'
    if 'Melanoma' in text:
        return 'MEL'
    if text.startswith('Benign::Benign melanocytic proliferations::Nevus'):
        return 'NV'
    return 'OTHER'


# 이 전체-pool 요약은 계약/EDA evidence용이다. 실제 class weight, prototype, tokenizer fit은 fold-local train에서만 해야 한다.
iddx_label_profile_df = df[['isic_id', 'patient_id', 'lesion_id', 'target', 'iddx_full']].copy()
iddx_label_profile_df['iddx_label_group'] = iddx_label_profile_df['iddx_full'].map(map_iddx_full_to_label_group)
iddx_label_profile_df['iddx_label_metric_group'] = iddx_label_profile_df['iddx_label_group'].where(
    iddx_label_profile_df['iddx_label_group'].isin(LABEL_METRIC_ACTIVE_GROUPS),
    pd.NA,
)
iddx_label_profile_df['iddx_label_is_metric_anchor'] = iddx_label_profile_df['iddx_label_group'].isin(LABEL_METRIC_ACTIVE_GROUPS)

label_group_summary_df = (
    iddx_label_profile_df.groupby('iddx_label_group', dropna=False)
    .agg(
        rows=('isic_id', 'size'),
        positives=('target', 'sum'),
        patients=('patient_id', 'nunique'),
        unique_iddx_full=('iddx_full', 'nunique'),
        metric_anchor_rows=('iddx_label_is_metric_anchor', 'sum'),
    )
    .reindex(LABEL_EMBEDDING_GROUPS)
)
label_group_summary_df['row_pct'] = (label_group_summary_df['rows'] / len(iddx_label_profile_df) * 100).round(4)
label_group_summary_df['positive_pct_in_group'] = (
    label_group_summary_df['positives'] / label_group_summary_df['rows'].replace(0, np.nan) * 100
).round(4)


# downstream candidate pipeline이 어떤 sidecar 컬럼을 어떤 scope에서 만들 수 있는지 문서화한다.
label_embedding_column_contract_df = pd.DataFrame(
    [
        {'column': 'iddx_label_group', 'values': 'MEL/BCC/SCC/NV/OTHER', 'role': '5-way train-only candidate label management unit', 'fit_scope': 'fixed rule; created only inside candidate pipeline after split assignment'},
        {'column': 'iddx_label_metric_group', 'values': 'MEL/BCC/SCC/NV or NA', 'role': 'active metric/prototype anchor group; OTHER masked', 'fit_scope': 'inner_train objectives only'},
        {'column': 'iddx_label_is_metric_anchor', 'values': 'True for MEL/BCC/SCC/NV, False for OTHER', 'role': 'supervised contrastive/triplet/prototype anchor mask', 'fit_scope': 'fixed mask; batch construction in inner_train only'},
        {'column': 'iddx_label_aux_weight', 'values': 'class-balanced for active groups; OTHER default 0.0-0.1', 'role': 'auxiliary CE/prototype loss weight', 'fit_scope': 'computed from inner_train only'},
    ]
)


# 수작업 예시로 매핑 규칙이 기대한 그룹을 반환하는지 최소 단위 검사를 남긴다.
mapping_unit_check_df = pd.DataFrame(
    [
        {'iddx_full_example': 'Malignant::Malignant adnexal epithelial proliferations - Follicular::Basal cell carcinoma', 'expected_group': 'BCC'},
        {'iddx_full_example': 'Malignant::Malignant epidermal proliferations::Squamous cell carcinoma in situ', 'expected_group': 'SCC'},
        {'iddx_full_example': 'Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma in situ::Melanoma in situ, associated with a nevus', 'expected_group': 'MEL'},
        {'iddx_full_example': 'Benign::Benign melanocytic proliferations::Nevus::Nevus, Atypical, Dysplastic, or Clark', 'expected_group': 'NV'},
        {'iddx_full_example': 'Benign', 'expected_group': 'OTHER'},
    ]
)
mapping_unit_check_df['mapped_group'] = mapping_unit_check_df['iddx_full_example'].map(map_iddx_full_to_label_group)
mapping_unit_check_df['pass'] = mapping_unit_check_df['mapped_group'] == mapping_unit_check_df['expected_group']
if not mapping_unit_check_df['pass'].all():
    raise AssertionError('iddx_full manual group mapping check failed')

display_table_with_note(
    '**표 5-a. iddx_full label group summary**',
    '이 표는 iddx_full 문자열을 후보 보조 라벨 그룹으로 묶었을 때의 분포를 보여준다. 이 요약은 candidate 설계용 evidence이며 실제 weight나 prototype은 fold-local train에서만 계산해야 한다.',
    label_group_summary_df,
)
display_table_with_note(
    '**표 5-b. label embedding candidate column contract**',
    '이 표는 iddx_full에서 만들 수 있는 candidate sidecar 컬럼과 사용 범위를 정리한다. fit_scope가 train objective로 제한된 항목은 validation, test, inference feature가 될 수 없다.',
    label_embedding_column_contract_df,
)
display_table_with_note(
    '**표 5-c. manual mapping unit check**',
    '이 표는 대표 iddx_full 문자열이 기대한 MEL, BCC, SCC, NV, OTHER 그룹으로 매핑되는지 확인한다. pass가 False이면 candidate 보조 라벨 규칙을 사용하기 전에 수정해야 한다.',
    mapping_unit_check_df,
)


**표 5-a. iddx_full label group summary**

이 표는 iddx_full 문자열을 후보 보조 라벨 그룹으로 묶었을 때의 분포를 보여준다. 이 요약은 candidate 설계용 evidence이며 실제 weight나 prototype은 fold-local train에서만 계산해야 한다.

,rows,positives,patients,unique_iddx_full,metric_anchor_rows,row_pct,positive_pct_in_group
iddx_label_group,,,,,,,
MEL,157,157,133,11,157,0.0391,100.0
BCC,163,163,110,4,163,0.0406,100.0
SCC,73,73,48,5,73,0.0182,100.0
NV,443,0,334,11,443,0.1105,0.0
OTHER,400223,0,1041,21,0,99.7916,0.0


**표 5-b. label embedding candidate column contract**

이 표는 iddx_full에서 만들 수 있는 candidate sidecar 컬럼과 사용 범위를 정리한다. fit_scope가 train objective로 제한된 항목은 validation, test, inference feature가 될 수 없다.

,column,values,role,fit_scope
0,iddx_label_group,MEL/BCC/SCC/NV/OTHER,5-way train-only candidate label management unit,fixed rule; created only inside candidate pipe...
1,iddx_label_metric_group,MEL/BCC/SCC/NV or NA,active metric/prototype anchor group; OTHER ma...,inner_train objectives only
2,iddx_label_is_metric_anchor,"True for MEL/BCC/SCC/NV, False for OTHER",supervised contrastive/triplet/prototype ancho...,fixed mask; batch construction in inner_train ...
3,iddx_label_aux_weight,class-balanced for active groups; OTHER defaul...,auxiliary CE/prototype loss weight,computed from inner_train only


**표 5-c. manual mapping unit check**

이 표는 대표 iddx_full 문자열이 기대한 MEL, BCC, SCC, NV, OTHER 그룹으로 매핑되는지 확인한다. pass가 False이면 candidate 보조 라벨 규칙을 사용하기 전에 수정해야 한다.

,iddx_full_example,expected_group,mapped_group,pass
0,Malignant::Malignant adnexal epithelial prolif...,BCC,BCC,True
1,Malignant::Malignant epidermal proliferations:...,SCC,SCC,True
2,Malignant::Malignant melanocytic proliferation...,MEL,MEL,True
3,Benign::Benign melanocytic proliferations::Nev...,NV,NV,True
4,Benign,OTHER,OTHER,True


### 5.1 해석 및 후속 구현 방향

Label embedding 후보는 baseline이 아니라 privileged supervision candidate다. `iddx_full`에서 만든 group, prototype, text representation, class weight는 fold-local `inner_train`에서만 계산해야 하며, `inner_validation`, `outer_test`, inference path에는 노출하지 않는다.


## 6. Patient-Level Triple Stratified Nested CV 계약

이 장은 split CSV를 저장하지 않고, `official_train_pool` 전체에서 논문 기본 protocol인 patient-level Triple Stratified Nested CV를 메모리에서 생성해 감사한다.

기본 구조는 다음과 같다.

```text
official_train_pool
  -> outer 5-fold Triple Stratified CV
  -> fold k: cv_test_fold == k 는 outer_test
  -> fold k: cv_test_fold != k 는 cv_train
  -> cv_train 내부에서 inner 4-fold Triple Stratified CV
  -> inner fold j: inner_validation
  -> 나머지 inner folds: inner_train
```

후속 구현에서 저장할 대표 artifact는 다음이다.

```text
data/splits/isic2024_official_train_nested_5x4_seed42.csv
columns: isic_id, patient_id, lesion_id, outer_fold, cv_test_fold, inner_fold, split_role
split_role: outer_test / inner_train / inner_validation
```


In [6]:
import time

SPLIT_SEED = 42
OUTER_FOLDS = 5
INNER_FOLDS = 4

nested_start_time = time.perf_counter()
print(
    f'Building patient-level Triple Stratified Nested CV: outer={OUTER_FOLDS}, inner={INNER_FOLDS}, '
    f'patients={df["patient_id"].nunique()}...',
    flush=True,
)
nested_result = build_nested_cv_assignments(
    df,
    seed=SPLIT_SEED,
    outer_folds=OUTER_FOLDS,
    inner_folds=INNER_FOLDS,
    sample_count_bins=5,
)
print(
    f'Nested CV built in {time.perf_counter() - nested_start_time:.1f}s '
    f'(outer_balance_score={nested_result.outer.balance_score:.6f})',
    flush=True,
)

patient_nested_profile_df = nested_result.patient_profile.copy()
patient_nested_profile_df['cv_test_fold'] = patient_nested_profile_df['patient_id'].astype(str).map(
    nested_result.outer.patient_assignment
).astype(int)
all_patient_set = set(patient_nested_profile_df['patient_id'].astype(str))

nested_terminology_df = pd.DataFrame(
    [
        {'term': 'official_train_pool', 'meaning': 'train-metadata.csv 전체 row; nested CV source'},
        {'term': 'cv_test_fold / outer_test', 'meaning': 'outer 5-fold CV에서 fold별 최종 평가 set; model/threshold 선택 금지'},
        {'term': 'cv_train', 'meaning': '현재 outer_test 환자를 제외한 outer training pool'},
        {'term': 'inner_validation', 'meaning': 'cv_train 내부 inner CV에서 model/hyperparameter/epoch/threshold/calibration 선택용 validation set'},
        {'term': 'inner_train', 'meaning': '현재 inner_validation 환자를 제외하고 model fit에 사용하는 set'},
    ]
)

inner_role_definition_df = pd.DataFrame(
    [
        {'split_role': 'inner_train', 'scope': 'within each outer cv_train and selected inner_fold', 'allowed_use': 'fit preprocessing, class weights, samplers, and model parameters'},
        {'split_role': 'inner_validation', 'scope': 'within each outer cv_train and selected inner_fold', 'allowed_use': 'model choice, hyperparameter, early stopping, threshold, and calibration only'},
        {'split_role': 'outer_test', 'scope': 'held-out patients for the current outer_fold', 'allowed_use': 'final evaluation only'},
    ]
)


def summarize_partition(patient_ids, split_role, outer_fold, inner_fold=pd.NA):
    rows = df.loc[df['patient_id'].astype(str).isin(patient_ids)]
    patient_summary = rows.groupby('patient_id')['target'].sum() if len(rows) else pd.Series(dtype=float)
    return {
        'outer_fold': outer_fold,
        'inner_fold': inner_fold,
        'split_role': split_role,
        'rows': int(len(rows)),
        'patients': int(rows['patient_id'].nunique()),
        'positive_rows': int(rows['target'].sum()),
        'positive_rate_pct': round(float(rows['target'].mean() * 100), 5) if len(rows) else 0.0,
        'patients_with_malignant': int((patient_summary > 0).sum()),
    }


outer_role_rows = []
outer_overlap_rows = []
outer_balance_rows = []
inner_role_rows = []
inner_overlap_rows = []
inner_balance_rows = []

for outer_fold in range(OUTER_FOLDS):
    outer_test_patients = set(
        patient_nested_profile_df.loc[
            patient_nested_profile_df['cv_test_fold'].eq(outer_fold),
            'patient_id',
        ].astype(str)
    )
    cv_train_patients = all_patient_set - outer_test_patients
    outer_role_rows.append(summarize_partition(cv_train_patients, 'cv_train', outer_fold))
    outer_role_rows.append(summarize_partition(outer_test_patients, 'cv_test_fold / outer_test', outer_fold))
    outer_overlap_rows.append(
        {
            'outer_fold': outer_fold,
            'cv_train_outer_test_patient_overlap': len(cv_train_patients & outer_test_patients),
            'pass': len(cv_train_patients & outer_test_patients) == 0,
        }
    )

    outer_fold_profile = patient_nested_profile_df.loc[
        patient_nested_profile_df['cv_test_fold'].eq(outer_fold)
    ]
    outer_balance_rows.append(
        {
            'outer_fold': outer_fold,
            'role': 'cv_test_fold / outer_test',
            'patients': int(len(outer_fold_profile)),
            'rows': int(outer_fold_profile['patient_rows'].sum()),
            'positive_rows': int(outer_fold_profile['positive_rows'].sum()),
            'malignant_patients': int(outer_fold_profile['has_malignant'].sum()),
            'sample_count_bins_present': int(outer_fold_profile['sample_count_bin'].nunique()),
            'triple_strata_present': int(outer_fold_profile['triple_stratum'].nunique()),
        }
    )

    inner_assignment = nested_result.inner_by_outer_fold[outer_fold].patient_assignment
    cv_train_profile = patient_nested_profile_df.loc[
        patient_nested_profile_df['patient_id'].astype(str).isin(cv_train_patients)
    ].copy()
    cv_train_profile['inner_fold'] = cv_train_profile['patient_id'].astype(str).map(inner_assignment).astype(int)
    for inner_fold in range(INNER_FOLDS):
        inner_validation_patients = set(
            cv_train_profile.loc[cv_train_profile['inner_fold'].eq(inner_fold), 'patient_id'].astype(str)
        )
        inner_train_patients = cv_train_patients - inner_validation_patients
        inner_role_rows.append(summarize_partition(inner_train_patients, 'inner_train', outer_fold, inner_fold))
        inner_role_rows.append(summarize_partition(inner_validation_patients, 'inner_validation', outer_fold, inner_fold))
        inner_overlap_rows.append(
            {
                'outer_fold': outer_fold,
                'inner_fold': inner_fold,
                'inner_train_inner_validation_patient_overlap': len(inner_train_patients & inner_validation_patients),
                'inner_train_outer_test_patient_overlap': len(inner_train_patients & outer_test_patients),
                'inner_validation_outer_test_patient_overlap': len(inner_validation_patients & outer_test_patients),
                'pass': not (
                    inner_train_patients & inner_validation_patients
                    or inner_train_patients & outer_test_patients
                    or inner_validation_patients & outer_test_patients
                ),
            }
        )
        for role, patient_ids in [('inner_train', inner_train_patients), ('inner_validation', inner_validation_patients)]:
            role_profile = cv_train_profile.loc[cv_train_profile['patient_id'].astype(str).isin(patient_ids)]
            inner_balance_rows.append(
                {
                    'outer_fold': outer_fold,
                    'inner_fold': inner_fold,
                    'role': role,
                    'patients': int(len(role_profile)),
                    'rows': int(role_profile['patient_rows'].sum()),
                    'positive_rows': int(role_profile['positive_rows'].sum()),
                    'malignant_patients': int(role_profile['has_malignant'].sum()),
                    'sample_count_bins_present': int(role_profile['sample_count_bin'].nunique()),
                    'triple_strata_present': int(role_profile['triple_stratum'].nunique()),
                }
            )

outer_role_summary_df = pd.DataFrame(outer_role_rows)
outer_patient_overlap_audit_df = pd.DataFrame(outer_overlap_rows)
outer_triple_balance_audit_df = pd.DataFrame(outer_balance_rows)
inner_role_summary_df = pd.DataFrame(inner_role_rows)
inner_patient_overlap_audit_df = pd.DataFrame(inner_overlap_rows)
inner_triple_balance_audit_df = pd.DataFrame(inner_balance_rows)

nested_split_artifact_contract_df = pd.DataFrame(
    [
        {'artifact': 'data/processed/isic2024_strict_model_input.csv', 'role': 'strict ordinary tabular feature table; split role not embedded'},
        {'artifact': 'data/processed/isic2024_iddx_full_train_only_sidecar.csv', 'role': 'candidate-only privileged sidecar; not ordinary input'},
        {'artifact': 'data/splits/isic2024_official_train_nested_5x4_seed42.csv', 'role': 'future nested CV assignment: outer_fold, inner_fold, split_role'},
    ]
)

nested_leakage_contract_df = pd.DataFrame(
    [
        {'contract': 'outer_patient_disjoint', 'rule': 'cv_train and cv_test_fold / outer_test patient_id overlap must be 0', 'review': '1차 outer split 생성 직후'},
        {'contract': 'inner_patient_disjoint', 'rule': 'inner_train, inner_validation, and outer_test patient_id overlap must be 0', 'review': '2차 inner split 생성 직후'},
        {'contract': 'triple_stratified_outer', 'rule': 'outer folds balance patients, rows, positive rows, malignant patients, sample-count bins, and triple strata', 'review': '1차 outer balance audit'},
        {'contract': 'triple_stratified_inner', 'rule': 'each outer cv_train is re-split with the same Triple Stratified objective', 'review': '2차 inner balance audit'},
        {'contract': 'runner_preflight_review', 'rule': 'Tabular/Image/Multimodal runners must read the same nested split artifact before training', 'review': '3차 runner preflight'},
        {'contract': 'result_summary_review', 'rule': 'result summary must record split_protocol, outer_fold, inner_fold, threshold_source, patient overlap audit, and triple balance audit', 'review': '4차 result summary audit'},
        {'contract': 'outer_test_final_only', 'rule': 'outer_test is not used for model choice, hyperparameter selection, early stopping, threshold selection, or calibration', 'review': 'all paper-facing runs'},
        {'contract': 'inference_no_iddx', 'rule': 'predict/inference API must not require iddx_full or iddx_label_group', 'review': 'baseline and candidate runs'},
    ]
)

if not outer_patient_overlap_audit_df['pass'].all():
    raise AssertionError('outer patient overlap audit failed')
if not inner_patient_overlap_audit_df['pass'].all():
    raise AssertionError('inner nested patient overlap audit failed')

display_table_with_note(
    '**표 6-a. Nested CV 용어 정의**',
    '이 표는 official_train_pool에서 시작하는 patient-level Triple Stratified Nested CV 용어를 고정한다. cv_test_fold는 outer_test와 같은 의미이며 최종 평가에만 사용한다.',
    nested_terminology_df,
)
display_table_with_note(
    f'**표 6-b. outer cv_test_fold / cv_train 분포: outer={OUTER_FOLDS}, seed={SPLIT_SEED}**',
    '이 표는 각 outer fold에서 cv_train과 cv_test_fold / outer_test의 row, patient, positive 분포를 확인한다. outer_test는 model 선택이나 threshold 선택에 사용하지 않는다.',
    outer_role_summary_df,
)
display_table_with_note(
    '**표 6-c. outer fold patient overlap audit**',
    '이 표는 각 outer fold에서 cv_train과 outer_test 사이 patient_id overlap이 0인지 확인한다. 하나라도 0이 아니면 paper-facing split으로 사용할 수 없다.',
    outer_patient_overlap_audit_df,
)
display_table_with_note(
    '**표 6-d. outer fold Triple Stratified balance audit**',
    '이 표는 outer_test fold들이 patient 수, row 수, positive row, malignant patient, sample-count bin, triple stratum 축을 유지하는지 확인한다.',
    outer_triple_balance_audit_df,
)
display_table_with_note(
    '**표 6-e. inner CV split role 요약**',
    '이 표는 nested CV 안에서 inner_train, inner_validation, outer_test가 어떤 목적으로 쓰이는지 고정한다. outer_test는 최종 평가 전용이고 선택 과정에는 들어가지 않는다.',
    inner_role_definition_df,
)
display_table_with_note(
    '**표 6-f. outer fold별 inner_train / inner_validation 분포**',
    '이 표는 각 outer cv_train 내부에서 다시 만든 inner_train과 inner_validation의 row, patient, positive 분포를 보여준다. inner_validation은 model choice, hyperparameter, epoch, threshold, calibration 선택용이다.',
    inner_role_summary_df,
)
display_table_with_note(
    '**표 6-g. inner_train / inner_validation / outer_test patient overlap audit**',
    '이 표는 모든 outer/inner 조합에서 inner_train, inner_validation, outer_test 사이 patient overlap이 0인지 확인한다. Nested CV의 핵심 누수 방지 감사다.',
    inner_patient_overlap_audit_df,
)
display_table_with_note(
    '**표 6-h. inner fold Triple Stratified balance audit**',
    '이 표는 각 outer cv_train 내부의 inner split도 Triple Stratified 성격을 유지하는지 확인한다. inner_validation도 patient-level grouped split이어야 한다.',
    inner_triple_balance_audit_df,
)
display_table_with_note(
    '**표 6-i. Nested CV leakage and review contract**',
    '이 표는 Nested CV를 최소 3회 이상 재검토하기 위한 감사 계약이다. split 생성 직후, inner split 생성 직후, runner preflight, result summary에서 반복 확인한다.',
    nested_leakage_contract_df,
)


Building patient-level Triple Stratified Nested CV: outer=5, inner=4, patients=1042...
Nested CV built in 74.8s (outer_balance_score=2.548586)


**표 6-a. Nested CV 용어 정의**

이 표는 official_train_pool에서 시작하는 patient-level Triple Stratified Nested CV 용어를 고정한다. cv_test_fold는 outer_test와 같은 의미이며 최종 평가에만 사용한다.

,term,meaning
0,official_train_pool,train-metadata.csv 전체 row; nested CV source
1,cv_test_fold / outer_test,outer 5-fold CV에서 fold별 최종 평가 set; model/thres...
2,cv_train,현재 outer_test 환자를 제외한 outer training pool
3,inner_validation,cv_train 내부 inner CV에서 model/hyperparameter/ep...
4,inner_train,현재 inner_validation 환자를 제외하고 model fit에 사용하는 set


**표 6-b. outer cv_test_fold / cv_train 분포: outer=5, seed=42**

...

이 표는 각 outer cv_train 내부의 inner split도 Triple Stratified 성격을 유지하는지 확인한다. inner_validation도 patient-level grouped split이어야 한다.

,outer_fold,inner_fold,role,patients,rows,positive_rows,malignant_patients,sample_count_bins_present,triple_strata_present
0,0,0,inner_train,615,240635,236,155,5,14
1,0,0,inner_validation,213,80212,79,52,5,13
2,0,1,inner_train,621,240635,236,155,5,14
3,0,1,inner_validation,207,80212,79,52,5,14
4,0,2,inner_train,623,240635,236,156,5,14
5,0,2,inner_validation,205,80212,79,51,5,14
6,0,3,inner_train,625,240636,237,155,5,14
7,0,3,inner_validation,203,80211,78,52,5,14
8,1,0,inner_train,617,240636,236,156,5,14
9,1,0,inner_validation,213,80211,79,51,5,14


**표 6-i. Nested CV leakage and review contract**

이 표는 Nested CV를 최소 3회 이상 재검토하기 위한 감사 계약이다. split 생성 직후, inner split 생성 직후, runner preflight, result summary에서 반복 확인한다.

,contract,rule,review
0,outer_patient_disjoint,cv_train and cv_test_fold / outer_test patient...,1차 outer split 생성 직후
1,inner_patient_disjoint,"inner_train, inner_validation, and outer_test ...",2차 inner split 생성 직후
2,triple_stratified_outer,"outer folds balance patients, rows, positive r...",1차 outer balance audit
3,triple_stratified_inner,each outer cv_train is re-split with the same ...,2차 inner balance audit
4,runner_preflight_review,Tabular/Image/Multimodal runners must read the...,3차 runner preflight
5,result_summary_review,"result summary must record split_protocol, out...",4차 result summary audit
6,outer_test_final_only,"outer_test is not used for model choice, hyper...",all paper-facing runs
7,inference_no_iddx,predict/inference API must not require iddx_fu...,baseline and candidate runs


### 6.1 해석

이 장의 핵심은 `cv_test_fold`와 `outer_test`가 같은 의미라는 점이다. 전체 `official_train_pool`을 patient-level Triple Stratified 5-fold로 나누고, 각 outer fold에서 현재 fold만 최종 평가용 `outer_test`로 둔다. 나머지 환자들은 `cv_train`이며, 여기까지만으로는 model choice나 threshold 선택을 하면 안 된다.

각 `cv_train`은 다시 patient-level Triple Stratified inner 4-fold로 나눈다. 특정 inner fold가 `inner_validation`이 되고, 나머지 inner folds가 `inner_train`이 된다. hyperparameter, model choice, early stopping epoch, threshold, calibration은 이 inner validation evidence에서만 정해야 한다.

따라서 모든 paper-facing 실험은 네 가지 분리를 동시에 만족해야 한다. 첫째, outer `cv_train`과 `outer_test` 사이 patient overlap이 0이어야 한다. 둘째, inner `inner_train`과 `inner_validation` 사이 patient overlap이 0이어야 한다. 셋째, inner set 어느 쪽도 outer_test patient를 포함하면 안 된다. 넷째, outer와 inner split 모두 malignant 희귀도와 환자별 sample 수 편차를 줄이기 위해 Triple Stratified balance audit을 통과해야 한다.

이 노트북은 아직 split CSV를 저장하지 않는다. 대신 재현의 시작점으로서 nested CV 계약과 감사표를 먼저 고정한다. 후속 CLI는 같은 규칙으로 `data/splits/isic2024_official_train_nested_5x4_seed42.csv`를 생성해야 하며, Tabular/Image/Multimodal runner는 같은 nested split artifact를 읽어야 한다.


## 7. handoff checklist

이 노트북은 Dataset class와 training pipeline 구현을 직접 수행하지 않는다. 다음 구현 단계에서 확인해야 할 계약을 넘긴다.


In [7]:
# baseline과 candidate experiment의 입력 계약을 downstream 구현자가 바로 확인할 수 있게 정리한다.
experiment_contract_df = pd.DataFrame(
    [
        {'experiment_group': 'strict_tabular_baseline', 'train_inputs': 'strict_input', 'inner_validation_outer_test_inference_inputs': 'strict_input', 'uses_iddx_full': False, 'paper_role': 'ordinary strict metadata baseline'},
        {'experiment_group': 'image_strict_multimodal_baseline', 'train_inputs': 'image + strict_input', 'inner_validation_outer_test_inference_inputs': 'image + strict_input', 'uses_iddx_full': False, 'paper_role': 'main multimodal baseline'},
        {'experiment_group': 'strict_input_iddx_full_candidate', 'train_inputs': 'strict_input + train-only iddx_full-derived auxiliary signal', 'inner_validation_outer_test_inference_inputs': 'strict_input only for scoring/inference', 'uses_iddx_full': 'train_only', 'paper_role': 'privileged supervision candidate, compare against strict baseline'},
    ]
)


# dataloader/model/predict API가 inference에서 iddx_full을 요구하지 않는지 구현 체크리스트로 고정한다.
implementation_checklist_df = pd.DataFrame(
    [
        {'item': 'Dataset item can return isic_id/patient_id/lesion_id/target/strict_input', 'required': True},
        {'item': 'train dataset may expose iddx_full_train_text or iddx_label_group for candidate branch', 'required': True},
        {'item': 'inner_validation, outer_test, and inference dataloaders work without iddx_full', 'required': True},
        {'item': 'model.forward/predict does not require iddx_full in inference mode', 'required': True},
        {'item': 'results record split_protocol, outer_fold, inner_fold, threshold_source, inference_requires_iddx_full=False', 'required': True},
        {'item': 'threshold source is inner_validation, not outer_test', 'required': True},
    ]
)


# threshold-dependent metric은 inner_validation에서 선택한 threshold만 사용해야 한다.
required_metrics_df = pd.DataFrame(
    [
        {'metric': 'pAUC@TPR>=0.80', 'required': True, 'threshold_dependent': False},
        {'metric': 'AUC', 'required': True, 'threshold_dependent': False},
        {'metric': 'F1', 'required': True, 'threshold_dependent': True},
        {'metric': 'precision', 'required': True, 'threshold_dependent': True},
        {'metric': 'recall', 'required': True, 'threshold_dependent': True},
        {'metric': 'balanced_accuracy', 'required': True, 'threshold_dependent': True},
        {'metric': 'Average Precision / PR-AUC', 'required': True, 'threshold_dependent': False},
    ]
)


display_table_with_note(
    '**표 7-a. baseline / candidate experiment contract**',
    '이 표는 strict baseline, multimodal baseline, iddx_full candidate의 입력 차이를 비교한다. candidate가 iddx_full을 쓰더라도 평가와 추론 입력은 strict baseline과 같아야 공정하다.',
    experiment_contract_df,
)
display_table_with_note(
    '**표 7-b. implementation handoff checklist**',
    '이 표는 후속 Dataset, DataLoader, model.forward, predict, result logging 구현에서 빠뜨리면 안 되는 요구사항이다. 특히 inference dataloader가 iddx_full을 요구하지 않아야 한다.',
    implementation_checklist_df,
)
display_table_with_note(
    '**표 7-c. required metric set**',
    '이 표는 ultra-rare malignant target에서 최소로 보고해야 할 metric set이다. Threshold-dependent metric은 outer_test가 아니라 inner_validation에서 고른 threshold로 계산해야 한다.',
    required_metrics_df,
)


**표 7-a. baseline / candidate experiment contract**

이 표는 strict baseline, multimodal baseline, iddx_full candidate의 입력 차이를 비교한다. candidate가 iddx_full을 쓰더라도 평가와 추론 입력은 strict baseline과 같아야 공정하다.

,experiment_group,train_inputs,inner_validation_outer_test_inference_inputs,uses_iddx_full,paper_role
0,strict_tabular_baseline,strict_input,strict_input,False,ordinary strict metadata baseline
1,image_strict_multimodal_baseline,image + strict_input,image + strict_input,False,main multimodal baseline
2,strict_input_iddx_full_candidate,strict_input + train-only iddx_full-derived au...,strict_input only for scoring/inference,train_only,"privileged supervision candidate, compare agai..."


**표 7-b. implementation handoff checklist**

이 표는 후속 Dataset, DataLoader, model.forward, predict, result logging 구현에서 빠뜨리면 안 되는 요구사항이다. 특히 inference dataloader가 iddx_full을 요구하지 않아야 한다.

,item,required
0,Dataset item can return isic_id/patient_id/les...,True
1,train dataset may expose iddx_full_train_text ...,True
2,"inner_validation, outer_test, and inference da...",True
3,model.forward/predict does not require iddx_fu...,True
4,"results record split_protocol, outer_fold, inn...",True
5,"threshold source is inner_validation, not oute...",True


**표 7-c. required metric set**

이 표는 ultra-rare malignant target에서 최소로 보고해야 할 metric set이다. Threshold-dependent metric은 outer_test가 아니라 inner_validation에서 고른 threshold로 계산해야 한다.

,metric,required,threshold_dependent
0,pAUC@TPR>=0.80,True,False
1,AUC,True,False
2,F1,True,True
3,precision,True,True
4,recall,True,True
5,balanced_accuracy,True,True
6,Average Precision / PR-AUC,True,False


### 7.1 최종 정리

이 노트북의 최종 결론은 다음과 같다.

- 메인 실험 입력은 `image + strict_input`이다.
- `train-metadata.csv` 전체를 `official_train_pool`로 사용한다.
- 기본 split은 patient-level Triple Stratified Nested CV다.
- `cv_test_fold`와 `outer_test`는 같은 의미이며 fold별 최종 평가 전용이다.
- `cv_train` 내부에서 다시 Triple Stratified inner CV를 만들고, `inner_train`과 `inner_validation`을 분리한다.
- `outer_test`는 model choice, threshold selection, calibration, early stopping에 사용하지 않는다.
- `inner_validation`은 반드시 `cv_train` 내부에서 patient-disjoint하게 만든다.
- `iddx_full`은 strict input이 아니라 candidate-only train-only sidecar다.
- 후속 export CLI와 baseline runner는 같은 nested split artifact를 읽고 `split_protocol`, `outer_fold`, `inner_fold`, `threshold_source`, `patient_overlap_audit`를 결과에 기록해야 한다.
